# Part 5 — Edge Case Handling and Tests

Verifies the edge cases required by the Week 8 assignment:

1. `order_items` contains an unknown `order_id`
2. `discount_percent > 100`
3. `quantity = 0`
4. `order_date` is in the future

It also verifies the frequently-bought-together query logic.

In [1]:
import sqlite3
from datetime import datetime, timedelta

def create_test_database():
    conn = sqlite3.connect(":memory:")

    conn.execute('''
        CREATE TABLE orders (
            order_id TEXT PRIMARY KEY,
            customer_id TEXT,
            order_date TEXT,
            status TEXT,
            region_code TEXT
        )
    ''')

    conn.execute('''
        CREATE TABLE order_items (
            item_id TEXT PRIMARY KEY,
            order_id TEXT,
            product_id TEXT,
            quantity INTEGER,
            unit_price REAL,
            discount_percent REAL
        )
    ''')

    conn.execute('''
        CREATE TABLE products (
            product_id TEXT PRIMARY KEY,
            product_name TEXT,
            category TEXT,
            subcategory TEXT,
            cost_price REAL
        )
    ''')

    return conn

## Test 1 — Unknown Order ID

In [2]:
def test_invalid_order_id():
    conn = create_test_database()

    conn.execute(
        "INSERT INTO orders VALUES (?, ?, ?, ?, ?)",
        ("ORD001", "CUST001", "2025-01-01 10:00:00",
         "DELIVERED", "WEST")
    )

    conn.execute(
        "INSERT INTO order_items VALUES (?, ?, ?, ?, ?, ?)",
        ("ITEM001", "ORD999", "PROD001", 2, 100, 10)
    )

    invalid_count = conn.execute('''
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN orders o
            ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
    ''').fetchone()[0]

    conn.close()

    assert invalid_count == 1
    print("✓ Test 1 passed: invalid order_id detected.")

## Test 2 — Discount Greater Than 100%

In [3]:
def test_discount_over_100():
    discount = 120

    assert discount > 100

    # Business rule: invalid discounts should be rejected.
    is_valid = 0 <= discount <= 100

    assert is_valid is False

    print(
        "✓ Test 2 passed: discount > 100% identified as invalid."
    )

## Test 3 — Quantity Equal to Zero

In [4]:
def test_zero_quantity():
    quantity = 0

    # Zero quantity should not contribute revenue.
    revenue = quantity * 100 * (1 - 10 / 100)

    assert revenue == 0

    print(
        "✓ Test 3 passed: quantity = 0 produces zero revenue."
    )

## Test 4 — Future Order Date

In [5]:
def test_future_order_date():
    today = datetime.now().date()
    future_date = today + timedelta(days=30)

    assert future_date > today

    print(
        "✓ Test 4 passed: future order date identified."
    )

## Test 5 — Frequently Bought Together

In [6]:
def test_frequently_bought_together():
    conn = create_test_database()

    products = [
        ("PROD001", "Laptop", "Electronics", "Laptop", 40000),
        ("PROD002", "Mouse", "Electronics", "Accessories", 500),
        ("PROD003", "Keyboard", "Electronics", "Accessories", 1000)
    ]

    conn.executemany(
        "INSERT INTO products VALUES (?, ?, ?, ?, ?)",
        products
    )

    orders = [
        ("ORD001", "CUST001", "2025-01-01 10:00:00", "DELIVERED", "WEST"),
        ("ORD002", "CUST002", "2025-01-02 10:00:00", "DELIVERED", "WEST"),
        ("ORD003", "CUST003", "2025-01-03 10:00:00", "DELIVERED", "WEST")
    ]

    conn.executemany(
        "INSERT INTO orders VALUES (?, ?, ?, ?, ?)",
        orders
    )

    items = [
        ("I1", "ORD001", "PROD001", 1, 40000, 0),
        ("I2", "ORD001", "PROD002", 1, 500, 0),

        ("I3", "ORD002", "PROD001", 1, 40000, 0),
        ("I4", "ORD002", "PROD002", 1, 500, 0),

        ("I5", "ORD003", "PROD001", 1, 40000, 0),
        ("I6", "ORD003", "PROD003", 1, 1000, 0)
    ]

    conn.executemany(
        "INSERT INTO order_items VALUES (?, ?, ?, ?, ?, ?)",
        items
    )

    result = conn.execute('''
        SELECT
            a.product_id AS product_a,
            b.product_id AS product_b,
            COUNT(DISTINCT a.order_id) AS times_bought_together
        FROM order_items a
        JOIN order_items b
            ON a.order_id = b.order_id
           AND a.product_id < b.product_id
        GROUP BY a.product_id, b.product_id
        ORDER BY times_bought_together DESC
    ''').fetchall()

    conn.close()

    assert result[0][0] == "PROD001"
    assert result[0][1] == "PROD002"
    assert result[0][2] == 2

    print(
        "✓ Test 5 passed: product pairs are counted correctly."
    )

## Run All Tests

In [7]:
test_invalid_order_id()
test_discount_over_100()
test_zero_quantity()
test_future_order_date()
test_frequently_bought_together()

print("\nAll Part 5 tests passed successfully.")

✓ Test 1 passed: invalid order_id detected.
✓ Test 2 passed: discount > 100% identified as invalid.
✓ Test 3 passed: quantity = 0 produces zero revenue.
✓ Test 4 passed: future order date identified.
✓ Test 5 passed: product pairs are counted correctly.

All Part 5 tests passed successfully.


## Edge-Case Handling Rules

In [8]:
print("Edge-case rules:")
print("1. Unknown order_id -> detected through referential-integrity check.")
print("2. discount_percent > 100 -> rejected as invalid.")
print("3. quantity = 0 -> contributes zero revenue.")
print("4. Future order_date -> identified as invalid/future data.")
print("5. Product pairs -> duplicate and same-product pairs are excluded.")

Edge-case rules:
1. Unknown order_id -> detected through referential-integrity check.
2. discount_percent > 100 -> rejected as invalid.
3. quantity = 0 -> contributes zero revenue.
4. Future order_date -> identified as invalid/future data.
5. Product pairs -> duplicate and same-product pairs are excluded.
